In [0]:
df = spark.table(
    "proyecto_final_mlops.telco_database.churn_dataset_training_data"
)

train_df, validation_df = df.randomSplit(
    [0.8, 0.2],
    seed=42
)

In [0]:
import pandas as pd

categorical_cols = [
    "Gender",
    "Subscription_Type",
    "Contract_Length"
]

numeric_cols = [
    "Age",
    "Tenure",
    "Usage_Frequency",
    "Support_Calls",
    "Payment_Delay",
    "Total_Spend",
    "Last_Interaction"
]

feature_cols = numeric_cols + categorical_cols

train_pd = train_df.select(feature_cols + ["Churn"]).toPandas()
validation_pd = validation_df.select(feature_cols + ["Churn"]).toPandas()

X_train = train_pd[feature_cols]
y_train = train_pd["Churn"]

X_validation = validation_pd[feature_cols]
y_validation = validation_pd["Churn"]

X_train = pd.get_dummies(X_train, columns=categorical_cols)
X_validation = pd.get_dummies(X_validation, columns=categorical_cols)

X_validation = X_validation.reindex(
    columns=X_train.columns,
    fill_value=0
)

In [0]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42)

In [0]:
pred_validation_rf = rf_model.predict(X_validation)
prob_validation_rf = rf_model.predict_proba(X_validation)[:, 1]

pred_train_rf = rf_model.predict(X_train)
prob_train_rf = rf_model.predict_proba(X_train)[:, 1]

In [0]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def calcular_metricas(y_real, pred, prob):
    return {
        "ROC_AUC": roc_auc_score(y_real, prob),
        "PR_AUC": average_precision_score(y_real, prob),
        "Accuracy": accuracy_score(y_real, pred),
        "Precision": precision_score(y_real, pred),
        "Recall": recall_score(y_real, pred),
        "F1": f1_score(y_real, pred)
    }

metricas_train_rf = calcular_metricas(
    y_train,
    pred_train_rf,
    prob_train_rf
)

metricas_validation_rf = calcular_metricas(
    y_validation,
    pred_validation_rf,
    prob_validation_rf
)

print("RANDOM FOREST - TRAIN")
print(metricas_train_rf)

print("\nRANDOM FOREST - VALIDATION")
print(metricas_validation_rf)

RANDOM FOREST - TRAIN
{'ROC_AUC': np.float64(1.0), 'PR_AUC': np.float64(1.0), 'Accuracy': 1.0, 'Precision': 1.0, 'Recall': 1.0, 'F1': 1.0}

RANDOM FOREST - VALIDATION
{'ROC_AUC': np.float64(0.9999992753003553), 'PR_AUC': np.float64(0.9999994181600749), 'Accuracy': 0.9992834150391847, 'Precision': 0.9999598054584188, 'Recall': 0.9987755183973342, 'F1': 0.9993673110720562}


In [0]:
comparacion_rf = spark.createDataFrame([
    ("Train 80%", *[float(v) for v in metricas_train_rf.values()]),
    ("Validation 20%", *[float(v) for v in metricas_validation_rf.values()])
], [
    "Dataset",
    "ROC_AUC",
    "PR_AUC",
    "Accuracy",
    "Precision",
    "Recall",
    "F1"
])

display(comparacion_rf)

Dataset,ROC_AUC,PR_AUC,Accuracy,Precision,Recall,F1
Train 80%,1.0,1.0,1.0,1.0,1.0,1.0
Validation 20%,0.9999992753003553,0.9999994181600749,0.9992834150391847,0.9999598054584188,0.9987755183973342,0.9993673110720562


In [0]:
import mlflow
import mlflow.sklearn

usuario = spark.sql("SELECT current_user()").first()[0]

experiment_name = f"/Users/{usuario}/customer_churn_random_forest"

mlflow.set_experiment(experiment_name)

with mlflow.start_run(run_name="random_forest_baseline") as run:

    mlflow.log_param("model", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("random_state", 42)

    for nombre, valor in metricas_validation_rf.items():
        mlflow.log_metric(
            f"validation_{nombre.lower()}",
            valor
        )

    run_id_rf = run.info.run_id

print("Run ID:", run_id_rf)

2026/09/07 02:19:12 INFO mlflow.tracking.fluent: Experiment with name '/Users/michaelarmasb@gmail.com/customer_churn_random_forest' does not exist. Creating a new experiment.


Run ID: 6bd2d1ff57fc45099069b52602511f22


In [0]:
from mlflow.models import infer_signature

signature = infer_signature(
    X_train,
    rf_model.predict(X_train)
)

with mlflow.start_run(
    run_id=run_id_rf
):

    mlflow.sklearn.log_model(
        rf_model,
        artifact_path="model",
        signature=signature,
        input_example=X_train.head(5)
    )

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/09/07 02:19:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-ebb9ad9e-bbdf.cloud.databricks.com/ml/experiments/3330361588753762/models/m-b6c0f8b2d64345628d559

In [0]:
mlflow.set_registry_uri("databricks-uc")

model_name_rf = (
    "proyecto_final_mlops."
    "telco_database."
    "churn_random_forest_model"
)

model_uri_rf = f"runs:/{run_id_rf}/model"

registered_model_rf = mlflow.register_model(
    model_uri=model_uri_rf,
    name=model_name_rf
)

print("Modelo:", model_name_rf)
print("Versión:", registered_model_rf.version)

Successfully registered model 'proyecto_final_mlops.telco_database.churn_random_forest_model'.
2026/09/07 02:19:35 WARNING mlflow.tracking._model_registry.fluent: Run with id 6bd2d1ff57fc45099069b52602511f22 has no artifacts at artifact path 'model', registering model based on models:/m-b6c0f8b2d64345628d5593035f04843a instead


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

Modelo: proyecto_final_mlops.telco_database.churn_random_forest_model
Versión: 1


🔗 Created version '1' of model 'proyecto_final_mlops.telco_database.churn_random_forest_model': https://dbc-ebb9ad9e-bbdf.cloud.databricks.com/explore/data/models/proyecto_final_mlops/telco_database/churn_random_forest_model/version/1?o=7474653079374067
